# 6 проектов -> таблица «Мониторинг показателей проектов»


Beta, Gamma, Delta, Epsilon, Zeta, Eta.
Каждому проекту соответствует пара файлов: <проект>_crm.xlsx + <проект>_pbx.xls

Формат АТС здесь — ПОЧАСОВОЙ ОТЧЁТ (24 строки + Итого),
а не реестр звонков. Отдельных звонков в нём нет.

Порядок работы в DataSpell:
    1. Положите свежие файлы в папку data/
    2. Ячейка 1 — настройки и импорт
    3. Ячейка 2 — сбор данных, проверка цифр на экране
    4. Ячейка 3 — выгрузка в Google Sheets


## ЯЧЕЙКА 1 — НАСТРОЙКИ

In [ ]:
import os
import re
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

import gspread

from etl_core import (
    DATA_DIR, CREDENTIALS, env, DEMO,
    DateResolver, ExcelReader, HistoryCache, SheetsLoader, Serializer,
    Validator, Palette, preview, log, RU_MONTHS, HOURS, TIME_LABELS,
)

BASE_DIR = str(DATA_DIR)   # выгрузки лежат в папке data/ в корне репозитория
# CREDENTIALS импортируется из etl_core: credentials.json в корне, в git не попадает
# Логин и пароль CRM — из .env рядом с проектом, тот же файл, что
# читают crm_reports.py и operators_report.ipynb:
#     CRM_USER=report_user
#     CRM_PASSWORD=пароль
load_dotenv(DATA_DIR.parent / ".env")  # .env лежит в корне репозитория

# etl_core обновляется вместе с ноутбуками, а Python держит уже
# загруженный модуль в памяти: заменить файл мало, нужен Restart Kernel.
# Проверяем сразу, иначе это выяснится на середине выгрузки.
_need = ["reset_sheet_format", "req_number_format", "batch_get",
         "list_date_sheets", "quote_sheet"]
_missing = [m for m in _need if not hasattr(SheetsLoader, m)]
if _missing:
    raise ImportError(
        "etl_core.py устарел — в нём нет методов: " + ", ".join(_missing)
        + "\n\n1. Положите свежий etl_core.py в папку etl/"
        + "\n2. Нажмите Restart Kernel — без этого Python продолжит "
          "использовать старую версию из памяти"
    )


# ID таблицы «Мониторинг показателей проектов» (из URL между /d/ и /edit).
# Эта таблица, в отличие от Alpha и Omega, НЕ месячная —
# листы всех месяцев лежат в ней же, поэтому ID постоянный.
# Если её когда-нибудь тоже разделят по месяцам, скажите —
# переведу на поиск по названию, как в alpha_registry.ipynb.
SPREADSHEET_ID = env("MONITORING_SHEET_ID")

# Проект -> (файл CRM, файл АТС)
# Порядок строк внутри каждого часа берётся отсюда
PROJECTS = {
    "Beta":    ("beta_crm.xlsx",    "beta_pbx.xls"),
    "Gamma":   ("gamma_crm.xlsx",   "gamma_pbx.xls"),
    "Delta":   ("delta_crm.xlsx",   "delta_pbx.xls"),
    "Epsilon": ("epsilon_crm.xlsx", "epsilon_pbx.xls"),
    "Zeta":    ("zeta_crm.xlsx",    "zeta_pbx.xls"),
    "Eta":     ("eta_crm.xlsx",     "eta_pbx.xls"),
}

COLUMNS = ["Часы", "Проекты", "IVR", "Входящие", "Потерянные", "Исходящие", "Итог"]

# --- колонка L «CRM/Входящие»: счёт с главного экрана CRM ---
# Считается не по выгруженному файлу, а живым запросом к CRM. Причина:
# выгрузка теряет первую запись запрошенного периода, и число из файла
# было бы занижено ровно на эту потерю. Колонка L существует для
# сверки с АТС, поэтому в ней нужна CRM как есть.
#
# False — не ходить в CRM, оставить L как была. Ставь, если CRM
# недоступна: остальная выгрузка от этого не зависит и отработает.
COUNT_CRM_INCOMING = True

CRM_IP = env("CRM_HOST", "crm.local")
CRM_BASE = f"http://{CRM_IP}/crm"

# Проект -> номер компании в CRM. Номера из выпадающего списка на
# странице «Отчёты», те же, что в crm_reports.py.
# ВНИМАНИЕ: в CRM бывают компании с похожими названиями — номер
# сверяй по списку, а не по имени.
CRM_COMPANY = {
    "Beta":    102,
    "Gamma":   103,
    "Delta":   104,
    "Epsilon": 105,
    "Zeta":    106,
    "Eta":     107,
}

# Пауза между запросами. Учётка общая с супервайзерами — сплошная
# очередь может сбросить им фильтры прямо во время работы.
CRM_PAUSE = 0.25

# Высота дневного листа. От неё же берётся верхняя граница диапазонов
# в формулах СУММЕСЛИ сводного блока — именно ОДНА константа на оба
# места, иначе они разъезжаются: формула, ссылающаяся за пределы
# листа, даёт #REF!, а формула с границей впритык под текущее число
# проектов перестаёт видеть данные, как только проект добавят.
# Запас: (400 - 2) / 24 часа = 16 проектов, сейчас их 6.
SHEET_ROWS = 400

# АТС для этих проектов всегда отдаёт готовый почасовой отчёт.
# Если придёт реестр звонков, скрипт остановится и покажет, что внутри
# файла — это защита от чужой выгрузки, сохранённой под другим именем.
# Ставить True, только если формат выгрузки сменился намеренно.
ALLOW_REGISTRY = False

# Что делать, если файла проекта нет в папке.
# True  — остановиться и показать список недостающих файлов.
# False — считать, что звонков не было: нули и предупреждение.
#         Ставьте False, только если проект правда не работал в этот день,
#         иначе в таблицу уйдут нули вместо настоящих цифр.
STOP_ON_MISSING_FILE = True

# Выгрузка CRM теряет первую запись периода. Для этих проектов из CRM
# берутся только ИСХОДЯЩИЕ, а первым звонком суток почти всегда идёт
# входящий — но «почти» не «всегда»: у мелких проектов (Epsilon,
# Delta, Zeta) ночного трафика нет, день начинается
# сразу с рабочих часов, и потерянный звонок вполне может оказаться
# исходящим. А исходящих у них 0-1 в день, то есть потеря одного —
# это вся дневная цифра.
#
# Поэтому CRM качаем за несколько дней (crm_reports.py берёт три). АТС —
# только за нужный день, как раньше: его почасовой отчёт по дням
# не делится. Скрипт берёт из CRM самую позднюю дату как рабочую,
# а более ранние показывает и отбрасывает.
USE_BUFFER_DAY = True

reader = ExcelReader(BASE_DIR)

# Операторы и уровень обслуживания в Google Sheets не попадают: на листах
# только почасовая сетка и сводный блок. Поэтому складываем их в тот же
# кэш, откуда аналитика берёт историю, — иначе по этим проектам её
# просто неоткуда взять.
CACHE_DIR = Path(BASE_DIR) / "cache"
cache_ops = HistoryCache(CACHE_DIR / "operators.csv")
cache_sl = HistoryCache(CACHE_DIR / "service_level.csv")


def _num_from_cell(v) -> float:
    """
    Число из ячейки АТС. Не распознано — NaN.

    Начиная с тысячи АТС ставит разделитель разрядов, и ячейка
    приходит как «1 036» — иногда с неразрывным пробелом. Обычный
    to_numeric на такой строке возвращает NaN, и сверка ломается
    ровно в тот день, когда проект перевалил за тысячу звонков.
    """
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return float("nan")
    s = (str(v).replace("\xa0", "").replace("\u2009", "")
         .replace(" ", "").replace(",", "."))
    if not s or s.lower() in ("nan", "none", "-"):
        return float("nan")
    return pd.to_numeric(s, errors="coerce")

## ЯЧЕЙКА 2 — СБОР ДАННЫХ

In [ ]:
def collect() -> tuple:
    """
    Собирает почасовую сетку по всем проектам.

    Возвращает (DataFrame отчёта, дата отчёта, DataFrame сверки).

    IVR / Входящие / Потерянные берутся из почасового отчёта АТС.
    Исходящие считаются по CRM: АТС исходящие не отдаёт.
    """
    frames = []
    checks = []
    skipped = []          # проекты, по которым сверить было не с чем
    empty_crm = []        # проекты с пустой выгрузкой CRM
    buffers = {}          # проект -> [(буферный день, строк), ...]
    ops_rows = []         # статистика операторов для кэша
    sl_rows = []          # уровень обслуживания по часам
    single_day = []       # проекты, где CRM за один день (буфера нет)
    no_files = []         # проекты, файлов которых нет в папке

    # Рабочая дата определяется ОДИН раз по всем файлам сразу, а не
    # каждым проектом отдельно. Иначе проект, у которого в этот день
    # звонков не было, взял бы дату вчерашнюю — и его вчерашние
    # исходящие попали бы в сегодняшний отчёт.
    crm_by_project, all_dates = {}, set()
    missing = []
    for project, (crm_file, inf_file) in PROJECTS.items():
        for f in (crm_file, inf_file):
            if not (Path(BASE_DIR) / f).exists():
                missing.append((project, f))
        try:
            c = reader.read_crm(crm_file)
        except FileNotFoundError:
            crm_by_project[project] = None
            continue
        c["_dt"] = pd.to_datetime(c["Время начала звонка"], errors="coerce")
        crm_by_project[project] = c
        all_dates |= set(c["_dt"].dt.date.dropna())

    if missing:
        lines = "\n".join(f"   {p:<20} {f}" for p, f in missing)
        if STOP_ON_MISSING_FILE:
            raise FileNotFoundError(
                f"В папке {BASE_DIR} не хватает файлов:\n{lines}\n\n"
                f"Скачайте их и запустите ячейку заново.\n"
                f"Если проект в этот день правда не работал — поставьте "
                f"STOP_ON_MISSING_FILE = False в ячейке 1, тогда по нему "
                f"будут записаны нули."
            )
        log.warning("Нет файлов, по этим проектам будут нули:\n%s", lines)

    if not all_dates:
        raise ValueError(
            "Ни в одном CRM-файле нет строк с датой. "
            "Проверьте, те ли файлы лежат в папке и не пустые ли они."
        )

    report_date = max(all_dates)
    earlier = sorted(x for x in all_dates if x < report_date)
    if earlier and not USE_BUFFER_DAY:
        log.warning("В файлах есть более ранние дни %s, но USE_BUFFER_DAY "
                    "выключен — они попадут в отчёт",
                    [x.strftime("%d.%m") for x in earlier])
    elif not earlier:
        single_day = list(PROJECTS)

    log.info("Рабочая дата по всем файлам: %s",
             report_date.strftime("%d.%m.%Y"))

    for project, (crm_file, inf_file) in PROJECTS.items():
        log.info("--- %s ---", project)

        crm = crm_by_project[project]
        d = report_date

        # Файла нет: АТС тоже читать нечем, поэтому по проекту
        # выходят нули. Предупреждение уже показано выше.
        if crm is None or not (Path(BASE_DIR) / inf_file).exists():
            no_files.append(project)
            block = pd.DataFrame({
                "Часы": [TIME_LABELS[h] for h in HOURS], "Проекты": project,
                "IVR": 0, "Входящие": 0, "Потерянные": 0, "Исходящие": 0,
                "_h": HOURS})
            block["Итог"] = 0
            frames.append(block)
            skipped.append(project)
            continue

        # Всё, что раньше рабочей даты, — буфер: именно у него откушена
        # первая запись, поэтому он и качался.
        if USE_BUFFER_DAY:
            older = crm[crm["_dt"].dt.date < d]
            if len(older):
                buffers.setdefault(project, [])
                for b in sorted(set(older["_dt"].dt.date)):
                    buffers[project].append(
                        (b, int((crm["_dt"].dt.date == b).sum())))
            crm = crm[crm["_dt"].dt.date == d]

        # У маленьких проектов выгрузка за день бывает пустой — звонков
        # просто не было. Это не ошибка: исходящие примем за ноль,
        # а АТС прочитается как обычно.
        if crm["_dt"].notna().any():
            out_mask = crm["Направление звонка"].astype(str).str.contains(
                "Исход", case=False, na=False
            )
            outbound = (crm[out_mask].dropna(subset=["_dt"])
                        .groupby(crm.loc[out_mask, "_dt"].dt.hour).size()
                        .reindex(HOURS, fill_value=0))

            # Операторы считаются по ВХОДЯЩИМ: исходящие — это обзвон,
            # а не приём обращений, и мешать их в одну кучу нельзя.
            inc = crm[~out_mask].dropna(subset=["_dt"])
            if len(inc) and "Оператор" in inc.columns:
                sec = pd.to_numeric(inc["Продолжительность, сек"],
                                    errors="coerce").fillna(0)
                names = inc["Оператор"].astype(str).str.strip()
                g = (pd.DataFrame({"Оператор": names, "_sec": sec.values})
                     .query("Оператор != '' and Оператор != 'nan'")
                     .groupby("Оператор", as_index=False)
                     .agg(Принято=("_sec", "size"), Секунды=("_sec", "sum")))
                for _, r in g.iterrows():
                    ops_rows.append({
                        "Дата": d, "Источник": project,
                        "Оператор": r["Оператор"],
                        "Принято": int(r["Принято"]),
                        "Секунды": float(r["Секунды"]),
                        "Звонков": int(r["Принято"])})
        else:
            log.warning("%s: за %s звонков в CRM нет — исходящие приму за 0",
                        project, d.strftime("%d.%m.%Y"))
            empty_crm.append(project)
            out_mask = pd.Series(False, index=crm.index)
            outbound = pd.Series(0, index=HOURS)

        # АТС: почасовые агрегаты
        inf = reader.read_pbx_hourly(inf_file,
                                     allow_registry=ALLOW_REGISTRY)
        inf = inf.set_index("Час").reindex(HOURS, fill_value=0)

        block = pd.DataFrame({
            "Часы": [TIME_LABELS[h] for h in HOURS],
            "Проекты": project,
            "IVR": inf["IVR"].values,
            "Входящие": inf["Входящие"].values,
            "Потерянные": inf["Потерянные"].values,
            "Исходящие": outbound.values,
            "_h": HOURS,
        })
        block["Итог"] = (block["IVR"] + block["Входящие"]
                         + block["Потерянные"] + block["Исходящие"])
        frames.append(block)

        # Уровень обслуживания: АТС отдаёт его готовым в колонке 15
        # почасового отчёта. В Google Sheets он не попадает, поэтому
        # забираем сюда — для аналитики это главный показатель качества.
        raw_sl = pd.read_excel(Path(BASE_DIR) / inf_file, header=None)
        if raw_sl.shape[1] > 15:
            body_sl = raw_sl[~raw_sl[0].astype(str).str.contains(
                "Итог", case=False, na=False)]
            for _, rr in body_sl.iterrows():
                m = re.match(r"^\s*(\d{1,2})[:.]", str(rr[0]))
                if not m:
                    continue
                v = _num_from_cell(rr[15])
                if pd.notna(v):
                    sl_rows.append({"Дата": d, "Источник": project,
                                    "Час": int(m.group(1)), "SL": float(v)})

        # Сверка с итоговой строкой самого АТС-файла.
        # Она есть только в почасовом отчёте; если АТС прислала
        # реестр звонков, сверять не с чем — предупреждаем явно,
        # чтобы пропуск не остался незамеченным.
        raw = pd.read_excel(Path(BASE_DIR) / inf_file, header=None)
        total_row = raw[raw[0].astype(str).str.contains("Итог", case=False, na=False)]
        if not total_row.empty:
            t = total_row.iloc[0]
            for col, idx in [("IVR", 1), ("Входящие", 2), ("Потерянные", 5)]:
                src = _num_from_cell(t[idx])
                agg = block[col].sum()
                if pd.isna(src):
                    # Пустая ячейка в «Итого» — это отсутствие эталона,
                    # а не эталон, равный нулю. Считать её нулём значит
                    # выдумать расхождение на ровном месте.
                    log.warning("%s / %s: в строке «Итого» пусто — "
                                "сверка по этому показателю пропущена",
                                project, col)
                    continue
                checks.append((f"{project} / {col}", int(agg), int(src)))
        else:
            skipped.append(project)
            log.warning("%s: в файле %s нет строки «Итого» — "
                        "сверка по этому проекту НЕ проводилась",
                        project, inf_file)

    # Сборка: сортировка по часу, внутри часа — порядок проектов из PROJECTS
    order = {p: i for i, p in enumerate(PROJECTS)}
    df = pd.concat(frames, ignore_index=True)
    df["_p"] = df["Проекты"].map(order)
    df = df.sort_values(["_h", "_p"]).drop(columns=["_h", "_p"]).reset_index(drop=True)

    if no_files:
        log.warning("ПО ЭТИМ ПРОЕКТАМ ЗАПИСАНЫ НУЛИ (файлов нет): %s",
                    ", ".join(no_files))
    if empty_crm:
        log.warning("Пустая выгрузка CRM: %s — проверьте, "
                    "правда ли звонков не было", ", ".join(empty_crm))
    if skipped:
        log.warning("Без сверки остались проекты: %s", ", ".join(skipped))

    return (df[COLUMNS], report_date, Validator.report(checks),
            buffers, single_day, pd.DataFrame(ops_rows), pd.DataFrame(sl_rows))


report, REPORT_DATE, checks, BUFFERS, SINGLE_DAY, OPS, SL = collect()

# Кэш пополняется сразу: аналитике эти данные больше взять негде
if not OPS.empty:
    cache_ops.merge(OPS)
    preview(OPS.groupby("Источник").agg(
        Операторов=("Оператор", "nunique"), Принято=("Принято", "sum")
    ).reset_index(), "ОПЕРАТОРЫ — СОХРАНЕНО В КЭШ")
if not SL.empty:
    cache_sl.merge(SL)
    log.info("Уровень обслуживания сохранён: %d записей", len(SL))

if BUFFERS:
    rows = [{"Проект": p, "Буферный день": b.strftime("%d.%m.%Y"),
             "Строк отброшено": n}
            for p, lst in BUFFERS.items() for b, n in lst]
    preview(pd.DataFrame(rows), "БУФЕРНЫЕ ДНИ — ОТБРОШЕНЫ, НЕ СЧИТАЮТСЯ")

if SINGLE_DAY:
    print("\nВ этих файлах CRM только один день, буфера нет — "
          "первый звонок суток потерян:")
    print(f"   {', '.join(SINGLE_DAY)}")
    print("   Если среди потерянных окажется исходящий, цифра будет занижена.")
    print("   Качайте CRM за несколько дней: буферные плюс нужный.")

preview(checks, "СВЕРКА С ИТОГАМИ АТС")
preview(report.head(15), f"ПОЧАСОВАЯ СЕТКА — {REPORT_DATE.strftime('%d.%m.%Y')} (первые 15 строк)")

totals = report[["IVR", "Входящие", "Потерянные", "Исходящие", "Итог"]].sum()
print(f"\nИТОГО ЗА ДЕНЬ: {totals.to_dict()}")
print(f"Строк в таблице: {len(report)} (должно быть "
          f"{24 * len(PROJECTS)} = 24 часа x {len(PROJECTS)} проектов)")

# Остановка при расхождении: не выгружаем заведомо кривые цифры
bad = checks[checks["OK"] == "✗"]
assert bad.empty, f"Расхождение с источником:\n{bad.to_string()}"

## ЯЧЕЙКА 3 — ВЫГРУЗКА В GOOGLE SHEETS

In [ ]:
def crm_login():
    """
    Входит в CRM и возвращает готовую сессию.

    Логин и пароль берутся из .env, в коде их нет.
    """
    user = os.environ.get("CRM_USER")
    password = os.environ.get("CRM_PASSWORD")
    if not user or not password:
        raise RuntimeError("Нет CRM_USER или CRM_PASSWORD — проверь .env")

    login_url = f"{CRM_BASE}/account/login"
    calls_url = f"{CRM_BASE}/calls/index"

    session = requests.Session()
    soup = BeautifulSoup(session.get(login_url, timeout=15).text, "html.parser")
    meta = soup.find("meta", {"name": "csrf-token"})
    if not meta or not meta.get("content"):
        raise RuntimeError("CSRF-токен не найден на странице входа")

    session.post(login_url, data={
        "_csrf": meta["content"],
        "LoginForm[username]": user,
        "LoginForm[password]": password,
    }, timeout=15)

    if "LoginForm" in session.get(calls_url, timeout=15).text:
        raise RuntimeError("Вход не удался — проверь логин и пароль в .env")
    log.info("CRM: вход выполнен под %s", user)
    return session


def crm_set_period(session, d) -> None:
    """
    Задаёт период на главном экране — это POST, а не параметр адреса.

    После него дата держится в сессии, поэтому дальше можно слать
    сколько угодно GET-запросов с фильтрами: период не слетит.
    """
    calls_url = f"{CRM_BASE}/calls/index"
    soup = BeautifulSoup(session.get(calls_url, timeout=15).text, "html.parser")
    meta = soup.find("meta", {"name": "csrf-token"})
    session.post(calls_url, params={"Reset": 1}, data={
        "_csrf": meta["content"] if meta else "",
        "period_start": f"{d:%Y-%m-%d} 00:00:00",
        "period_end": f"{d:%Y-%m-%d} 23:59:59",
    }, timeout=30)


def crm_count_incoming(session, company_id: int) -> int:
    """
    Сколько входящих звонков у компании за уже заданный период.

    Число берётся из строки «Показаны 1-20 из N записи» — CRM считает
    его сама, качать строки незачем. Тысячи там разделены пробелом
    («4 971»), поэтому оставляем только цифры.

    Фильтр направления отправляем всегда; пустое значение здесь НЕ
    игнорируется, а фильтрует по пустому — это уже ловилось раньше,
    поэтому значение «0» (Входящий) передаётся явно.
    """
    resp = session.get(f"{CRM_BASE}/calls/index", params={
        "CallSearch[CompanyID]": str(company_id),
        "CallSearch[Direction]": "0",
    }, timeout=30)
    resp.raise_for_status()
    time.sleep(CRM_PAUSE)

    el = BeautifulSoup(resp.text, "html.parser").find(class_="summary")
    if not el:
        return 0
    m = re.search(r"из\s+([\d\s\u00a0]+?)\s*запис",
                  el.get_text(" ", strip=True))
    return int(re.sub(r"\D", "", m.group(1))) if m else 0


def crm_incoming_by_project(d) -> dict:
    """
    Входящие по каждому проекту за день — для колонки L.

    Возвращает {проект: число}. Если CRM недоступна или вход не удался,
    возвращает пустой словарь и пишет предупреждение: колонка L просто
    не обновится, а вся остальная выгрузка — из файлов — не зависит от
    CRM и должна отработать как обычно.
    """
    if not COUNT_CRM_INCOMING:
        return {}
    if DEMO:
        log.info("Демо-режим: в CRM не хожу — колонка «CRM/Входящие» "
                 "останется пустой")
        return {}

    try:
        session = crm_login()
        crm_set_period(session, d)
    except Exception as e:
        log.warning("CRM недоступна (%s: %s) — колонка «CRM/Входящие» "
                    "не обновлена, остальное выгружено", type(e).__name__, e)
        return {}

    out = {}
    for project in PROJECTS:
        company_id = CRM_COMPANY.get(project)
        if company_id is None:
            log.warning("Для проекта «%s» не задан номер компании в "
                        "CRM_COMPANY — пропускаю", project)
            continue
        try:
            out[project] = crm_count_incoming(session, company_id)
        except Exception as e:
            log.warning("«%s»: не удалось посчитать входящие (%s)",
                        project, type(e).__name__)
    return out


def summary_block(crm_inc: dict = None) -> list:
    """
    Собирает сводный блок I:O для нового листа.

    Раскладка повторяет текущие листы:
        I Проекты | J IVR | K Входящие | L CRM/Входящие
        M Потерянные | N Исходящие | O Итог

    J, K, M, N — формулы СУММЕСЛИ по колонке «Проекты» (B).
    L — живой счёт входящих из CRM, приходит готовым в crm_inc.
    O — сумма показателей строки, без учёта L.

    Почему L заполняется ЗДЕСЬ, а не отдельной записью: блок
    пишется одним диапазоном I1:O(n+1), который накрывает и колонку
    L. Если писать L отдельно, а блок следом — блок затрёт её
    пустотой. Ровно так и было: на новых листах колонка оставалась
    пустой, на старых (где блок не переписывается) — заполнялась.

    Диапазон суммирования — до SHEET_ROWS, то есть по всей высоте
    листа, а не впритык под текущее число проектов. СУММЕСЛИ ищет по
    ТЕКСТУ в колонке B, поэтому пустые строки внизу ей не мешают: не
    найдя совпадения, они ничего не добавляют к сумме. Строка «Итого»
    тоже безопасна — в колонке B у неё слово «Итого», а не название
    проекта, под критерий она не подходит.

    Почему именно SHEET_ROWS, а не произвольное большое число: если
    формула сошлётся ЗА пределы листа, Google Sheets вернёт #REF! и
    сломает весь блок. А если граница будет впритык под текущее число
    проектов — она застынет в тексте формулы с датой создания листа,
    и при добавлении проекта данные, легшие ниже, молча выпадут из
    суммы. Ровно это и случилось при переходе с 5 проектов на 6.
    """
    header = ["Проекты", "IVR", "Входящие", "CRM/Входящие",
              "Потерянные", "Исходящие", "Итог"]
    rows = [header]

    # Колонки-источники в левой таблице: C=IVR, D=Входящие,
    # E=Потерянные, F=Исходящие
    src = {"J": "C", "K": "D", "M": "E", "N": "F"}
    last = SHEET_ROWS

    for i, project in enumerate(PROJECTS):
        r = i + 2                           # строка блока в Sheets
        line = [project]
        for col in ["J", "K"]:
            line.append(f"=СУММЕСЛИ($B$2:$B${last};$I{r};"
                        f"${src[col]}$2:${src[col]}${last})")
        # L — CRM/Входящие. Нет числа (CRM недоступна) — пустая
        # строка, чтобы не затереть то, что могло стоять раньше.
        line.append((crm_inc or {}).get(project, ""))
        for col in ["M", "N"]:
            line.append(f"=СУММЕСЛИ($B$2:$B${last};$I{r};"
                        f"${src[col]}$2:${src[col]}${last})")
        # Итог: IVR + Входящие + Потерянные + Исходящие, без CRM/Входящие
        line.append(f"=J{r}+K{r}+M{r}+N{r}")
        rows.append(line)

    return rows


def upload(df: pd.DataFrame, d) -> None:
    """
    Выгружает почасовую сетку на лист с датой и применяет форматирование.

    Раскладка листа повторяет текущий вид таблицы:
      A       Часы (объединено по строкам каждого часа —
              по числу проектов, len(PROJECTS))
      B       Проекты
      C:G     IVR, Входящие, Потерянные, Исходящие, Итог
      строка n+2  Итого — формулой =SUM() (n = проекты x 24 часа)
    """
    loader = SheetsLoader(CREDENTIALS, SPREADSHEET_ID).connect()

    # Очищается ТОЛЬКО диапазон A:G. Блок I:O трогать нельзя —
    # там формулы =СУММЕСЛИ, которые считают итоги по проектам сами.
    loader.open_sheet_for_date(d, rows=SHEET_ROWS, cols=20,
                               clear_range=f"A1:G{SHEET_ROWS}")

    n = len(df)                      # проекты x 24 часа строк данных
    total_row_idx = n + 2            # номер строки «Итого» в Sheets (1-based)

    # Значения. Колонка «Часы» заполняется только в первой строке каждого
    # часа — остальные ячейки пустые, они уйдут под объединение.
    values = Serializer.frame(df, header=True)
    for i in range(1, len(values)):
        if (i - 1) % len(PROJECTS) != 0:
            values[i][0] = ""

    # Строка «Итого» — формулы, чтобы пересчитывалась при ручной правке
    total = ["", "Итого"] + [
        f"=SUM({c}2:{c}{n + 1})" for c in ["C", "D", "E", "F", "G"]
    ]

    payload = [
        {"range": "A1", "values": values},
        {"range": f"A{total_row_idx}", "values": [total]},
    ]

    # Колонка L «CRM/Входящие» — живой счёт с главного экрана CRM.
    crm_inc = crm_incoming_by_project(d)
    if crm_inc:
        log.info("CRM/Входящие: %s",
                 ", ".join(f"{p} {crm_inc[p]}" for p in PROJECTS
                          if p in crm_inc))

    # Сводный блок I:O заполняется ТОЛЬКО на новом листе — на
    # существующих там могут быть ручные правки, и затирать их нельзя.
    #
    # ВАЖНО: это НЕ значит, что старые листы всегда в порядке. Формула
    # это текст, а не код: записанная один раз, она не перечитывает
    # настройки. Если число проектов изменится, на уже созданных
    # листах диапазоны останутся прежними и новый проект в сумму не
    # попадёт. Для таких случаев был отдельный одноразовый скрипт (в
    # репозиторий не входит): он переписывал только J,K,M,N на нужных днях.
    if loader.sheet_created:
        # Новый лист: L идёт внутри блока, отдельной записи нет —
        # иначе блок затёр бы её пустотой.
        payload.append({"range": "I1", "values": summary_block(crm_inc)})
        log.info("Лист новый — записан сводный блок I:O")
    else:
        # Существующий лист: блок не трогаем (там бывают ручные
        # правки), поэтому L дописываем отдельно. Наложения нет —
        # именно потому, что блок в этой ветке не пишется.
        log.info("Лист существующий — блок I:O не тронут")
        if crm_inc:
            payload.append({
                "range": f"L2:L{len(PROJECTS) + 1}",
                "values": [[crm_inc.get(p, "")] for p in PROJECTS],
            })
            log.info("Колонка L обновлена отдельной записью")

    loader.write(payload)

    # --- форматирование ---
    reqs = [
        loader.req_header(0, 0, 7, Palette.HEADER_GREEN),
        loader.req_freeze(1),
        loader.req_col_width(0, 1, 110),    # Часы
        loader.req_col_width(1, 2, 140),    # Проекты
        loader.req_col_width(2, 7, 90),     # метрики
        loader.req_borders(0, total_row_idx, 0, 7),
        loader.req_center(1, n + 1, 2, 7),  # числа по центру
        loader.req_row_fill(total_row_idx - 1, 0, 7, Palette.TOTAL_YELLOW, bold=True),
        loader.req_center(total_row_idx - 1, total_row_idx, 0, 7, bold=True),
    ]

    # Объединение ячеек часа: по одному блоку на каждый час
    step = len(PROJECTS)
    for i in range(24):
        r = 1 + i * step
        reqs.append(loader.req_merge(r, r + step, 0, 1))
    reqs.append(loader.req_center(1, n + 1, 0, 1))

    # Проекты жирным — как в текущей таблице
    reqs.append({
        "repeatCell": {
            "range": {"sheetId": loader._sheet_id(),
                      "startRowIndex": 1, "endRowIndex": n + 1,
                      "startColumnIndex": 1, "endColumnIndex": 2},
            "cell": {"userEnteredFormat": {
                "textFormat": {"bold": True, "fontSize": 10}}},
            "fields": "userEnteredFormat.textFormat",
        }
    })

    # Оформление сводного блока — только если он был записан
    if loader.sheet_created:
        n_proj = len(PROJECTS)
        reqs += [
            loader.req_header(0, 8, 15, Palette.HEADER_GREEN),
            loader.req_borders(0, n_proj + 1, 8, 15),
            loader.req_center(1, n_proj + 1, 9, 15),
            loader.req_col_width(8, 9, 140),
            loader.req_col_width(9, 15, 110),
        ]

    loader.apply_format(reqs, strict=False)
    print(f"\nГотово: лист '{DateResolver.sheet_name(d)}' в «{loader.sh.title}»")


upload(report, REPORT_DATE)

## ЯЧЕЙКА 4 — ИТОГ ЗА МЕСЯЦ

In [ ]:
# В демо-режиме итог месяца не считается. Он складывает значения формул
# СУММЕСЛИ с дневных листов, а демо-Excel хранит формулы текстом и не
# считает их: сверка блока с сеткой показала бы расхождение, которого нет.
if DEMO:
    raise RuntimeError("Демо-режим: итог за месяц не считается — ему нужны "
                       "посчитанные формулы Google Sheets. Дневной лист "
                       "записан ячейкой 3.")

# Как называется лист с месячным итогом. Месяц подставляется из даты
# файла, поэтому в августе править ничего не нужно.
MONTH_TOTAL_SHEET = "Итог {month}"

# Колонка D «CRM/Входящие» складывается из колонок L дневных листов.
#
# Раньше здесь стояло False: колонка L заполнялась вручную, и скрипт
# не имел права затирать вписанное человеком. Теперь L заполняется
# сама — живым счётом с главного экрана CRM (см. COUNT_CRM_INCOMING
# в настройках), и запрет потерял смысл.
#
# ВАЖНО про неполные месяцы: итог складывает то, что реально лежит
# в листах. Если в каких-то днях месяца L пустая — лист создан до
# появления этой колонки или CRM тогда была недоступна — сумма выйдет
# заниженной, и помечена как частичная она не будет. Прежде чем
# верить месячному числу, стоит прогнать monitoring.ipynb по тем дням
# заново: L заполнится, и итог станет полным.
WRITE_CRM_COLUMN = True

# Колонку G «Итог» скрипт не пишет никогда: там формула, которая
# складывает строку сама. Записать в неё число значит убить формулу.

# Позиции в блоке I:O листа дня совпадают с A:G листа итога:
# 0 Проекты, 1 IVR, 2 Входящие, 3 CRM/Входящие,
# 4 Потерянные, 5 Исходящие, 6 Итог
METRIC_NAMES = ["IVR", "Входящие", "CRM/Входящие",
                "Потерянные", "Исходящие", "Итог"]


def _month_bounds(d):
    """Первый и последний день месяца, в котором лежит дата."""
    import calendar
    return d.replace(day=1), d.replace(day=calendar.monthrange(d.year, d.month)[1])


def collect_month(d) -> tuple:
    """
    Складывает блоки I:O со всех листов месяца.

    Возвращает (суммы по проектам, отчёт о собранных днях).
    Значения приходят из Sheets строками и с разделителем разрядов,
    поэтому разбираются той же функцией, что и сверка.
    """
    loader = SheetsLoader(CREDENTIALS, SPREADSHEET_ID).connect()
    d_from, d_to = _month_bounds(d)
    sheets = loader.list_date_sheets(d_from, d_to)

    if not sheets:
        raise ValueError(
            f"В таблице нет ни одного листа за {d.strftime('%m.%Y')}. "
            f"Проверьте имена вкладок — они должны быть вида 01.07.26"
        )

    # Вкладки, похожие на дату, но не разобранные: чаще всего опечатка
    # в имени. Формула такие дни молча теряет, скрипт — называет вслух.
    # Вкладка, похожая на дату, но не попавшая в выборку, — это либо
    # опечатка в имени, либо просто другой месяц. Второе — норма, и
    # пугать этим не нужно, поэтому разбираем отдельно: пробуем
    # прочитать имя как дату и смотрим, совпал ли месяц.
    known = set(sheets.values())
    suspicious = []
    for w in loader.sh.worksheets():
        t = w.title.strip()
        if t in known or t.lower().startswith("итог"):
            continue
        if not re.match(r"^\d{1,2}[.\-/]\d{1,2}", t):
            continue
        parsed = None
        for fmt in ("%d.%m.%y", "%d.%m.%Y", "%d-%m-%y", "%d/%m/%y"):
            try:
                parsed = datetime.strptime(t, fmt).date()
                break
            except ValueError:
                continue
        # Имя читается как дата другого месяца — всё в порядке,
        # молчим. Не читается вовсе — вот это стоит показать.
        if parsed is None:
            suspicious.append(t)

    q = loader.quote_sheet
    # Читаем два источника сразу: сводный блок I:O и почасовую сетку A:G.
    # Блок собран формулами СУММЕСЛИ по сетке, поэтому их итоги обязаны
    # совпасть. Расходятся — значит формула на каком-то листе сбита
    # или в сетку попала лишняя строка; молча такое пропускать нельзя.
    ranges, kinds = [], []
    for t in sheets.values():
        # Высоту сетки берём из числа проектов, а не фиксированным
        # числом. Раньше стояло A1:G130: при пяти проектах сетка была
        # 120 строк и влезала, а с шестым стала 144 — всё, что ниже
        # 130-й, при сверке не читалось. Формулы СУММЕСЛИ считали
        # честно, проверка — нет, и блок всегда выходил «больше».
        # Лишние пустые строки разбору не мешают: строки без имени
        # проекта и строка «Итого» пропускаются ниже по имени.
        ranges += [f"{q(t)}!I1:O12", f"{q(t)}!A1:G{SHEET_ROWS}"]
        kinds += ["block", "grid"]
    got = []
    for i in range(0, len(ranges), 25):
        got += loader.batch_get(ranges[i:i + 25])
    grids = [g for g, k in zip(got, kinds) if k == "block"]
    hourly_grids = [g for g, k in zip(got, kinds) if k == "grid"]

    # Значения ошибок формул: пустая ячейка и «#ССЫЛКА!» для суммы
    # одинаково дают ноль, но означают разное. Первое — данных нет,
    # второе — формула сломана, и день молча выпадет из итога.
    ERRORS = ("#", "#ССЫЛКА", "#ЗНАЧ", "#ДЕЛ", "#ИМЯ", "#Н/Д", "#ПУСТО",
              "#ЧИСЛО", "#REF", "#VALUE", "#DIV", "#NAME", "#N/A", "#NULL", "#NUM")

    totals, per_day, empty_days, broken = {}, {}, [], []
    for day, grid in zip(sheets.keys(), grids):
        rows_seen = 0
        day_vals = {}
        for row in (grid or [])[1:]:            # первая строка — шапка
            if not row:
                continue
            name = str(row[0]).strip()
            if not name or name.lower().startswith("итог"):
                continue
            vals = totals.setdefault(name, [0.0] * 6)
            got_row = []
            for i in range(6):
                cell = row[i + 1] if i + 1 < len(row) else None
                text = str(cell).strip().upper() if cell is not None else ""
                if text.startswith("#"):
                    broken.append((day, name, METRIC_NAMES[i], str(cell)))
                    got_row.append(np.nan)
                    continue
                v = _num_from_cell(cell)
                got_row.append(v)
                if pd.notna(v):
                    vals[i] += v
            day_vals[name] = got_row
            rows_seen += 1
        per_day[day] = {"строк": rows_seen, "значения": day_vals}
        if not rows_seen:
            empty_days.append(day)

    # Дни месяца, для которых листа вообще нет. Будущие дни не в счёт.
    last = min(d_to, d)
    expected = pd.date_range(d_from, last).date
    missing_days = [x for x in expected if x not in sheets]

    # Контрольный пересчёт по почасовой сетке: колонка B — проект,
    # C..F — IVR, входящие, потерянные, исходящие
    control = {}
    for grid in hourly_grids:
        for row in (grid or [])[1:]:
            if len(row) < 2:
                continue
            name = str(row[1]).strip()
            if not name or name.lower().startswith("итог"):
                continue
            v = control.setdefault(name, [0.0] * 4)
            for i in range(4):
                x = _num_from_cell(row[i + 2]) if i + 2 < len(row) else float("nan")
                if pd.notna(x):
                    v[i] += x

    mismatch = []
    for name, vals in totals.items():
        c = control.get(name)
        if c is None:
            continue
        # В блоке порядок IVR, Входящие, CRM/Входящие, Потерянные, Исходящие
        pairs = [("IVR", vals[0], c[0]), ("Входящие", vals[1], c[1]),
                 ("Потерянные", vals[3], c[2]), ("Исходящие", vals[4], c[3])]
        for metric, a, b in pairs:
            if abs(a - b) > 0.001:
                mismatch.append((name, metric, int(a), int(b)))

    report = {
        "листов найдено": len(sheets),
        "с данными": len(sheets) - len(empty_days),
        "пустых": [d_.strftime("%d.%m") for d_ in empty_days],
        "нет листа": [x.strftime("%d.%m") for x in missing_days],
        "не распознаны": suspicious,
        "расхождения": mismatch,
        "сломанные формулы": broken,
    }
    return totals, report, loader, d_from, d_to, per_day


def _find_month_block(grid: list, month: str) -> tuple:
    """
    Находит на листе блок нужного месяца.

    Лист итогов держит несколько месяцев подряд: строка с названием
    («Июль»), под ней шапка «Проекты | IVR | ...», затем строки проектов,
    и дальше следующий месяц. Названия проектов в блоках повторяются,
    поэтому искать их по всему листу нельзя — июльские цифры уедут
    в августовский блок.

    Возвращает (номер строки шапки, [(номер строки, проект), ...]).
    Строки нумеруются от 1, как в Sheets. Блока нет — (None, []).
    """
    months = set(RU_MONTHS.values())
    start = None
    for i, row in enumerate(grid):
        val = str(row[0]).strip() if row and row[0] is not None else ""
        if val == month:
            start = i
            break
    if start is None:
        return None, []

    # Шапка идёт сразу под названием месяца
    head_idx = start + 1
    if head_idx >= len(grid):
        return None, []

    rows = []
    for i in range(head_idx + 1, len(grid)):
        row = grid[i]
        val = str(row[0]).strip() if row and row[0] is not None else ""
        if not val:
            continue
        if val in months:            # начался следующий месяц
            break
        rows.append((i + 1, val))    # +1: нумерация Sheets с единицы
    return head_idx + 1, rows


def write_month_total(totals: dict, loader, d) -> pd.DataFrame:
    """
    Заполняет блок месяца на листе итогов.

    Значения подставляются НАПРОТИВ названий, которые уже есть в
    колонке A внутри блока этого месяца, — так сохраняется порядок
    строк и не затирается проект, по которому данных нет.

    Колонка G («Итог») не пишется никогда: там формула строки.
    Колонка D («CRM/Входящие») — только при WRITE_CRM_COLUMN.
    """
    month = RU_MONTHS[d.month]

    # Лист итогов один на все месяцы. Ищем любой лист, чьё имя
    # начинается с «Итог», и уже внутри него — блок нужного месяца.
    ws = None
    for w in loader.sh.worksheets():
        if w.title.strip().lower().startswith("итог"):
            ws = w
            break
    if ws is None:
        title = MONTH_TOTAL_SHEET.format(month=month)
        ws = loader.sh.add_worksheet(title=title, rows=60, cols=10)
        ws.batch_update([
            {"range": "A1", "values": [[month]]},
            {"range": "A2", "values": [["Проекты"] + METRIC_NAMES]},
            {"range": "A3", "values": [[p] for p in PROJECTS]},
            {"range": "G3", "values": [[f"=B{i}+C{i}+E{i}+F{i}"]
                                       for i in range(3, len(PROJECTS) + 3)]},
        ], value_input_option="USER_ENTERED")
        log.info("Создан лист «%s» с блоком «%s»", title, month)
    loader.ws = ws

    grid = ws.get("A1:G200") or []
    head_row, names = _find_month_block(grid, month)

    if head_row is None:
        # Месяца на листе ещё нет — дописываем блок снизу
        first = len(grid) + 2
        block = [[month]] + [["Проекты"] + METRIC_NAMES] + [[p] for p in PROJECTS]
        ws.batch_update([
            {"range": f"A{first}", "values": block},
            {"range": f"G{first + 2}",
             "values": [[f"=B{i}+C{i}+E{i}+F{i}"]
                        for i in range(first + 2, first + 2 + len(PROJECTS))]},
        ], value_input_option="USER_ENTERED")
        log.info("На листе «%s» добавлен блок «%s» со строки %d",
                 ws.title, month, first)
        grid = ws.get("A1:G200") or []
        head_row, names = _find_month_block(grid, month)

    if not names:
        raise ValueError(
            f"На листе «{ws.title}» не нашлись строки проектов "
            f"в блоке «{month}». Проверьте, что под названием месяца "
            f"идёт шапка «Проекты», а под ней — названия проектов."
        )

    log.info("Блок «%s»: шапка в строке %d, проектов %d",
             month, head_row, len(names))

    # Обратная проверка: есть посчитанный итог, а строки под него в
    # блоке нет. Так будет с любым новым проектом в первый месяц
    # после добавления — блок «Итог» на этот месяц уже создан со
    # старым числом строк, и просто добавить проект в PROJECTS
    # недостаточно: он не появится в готовом блоке сам. Раньше это
    # проходило совсем тихо — посчитанные деньги проекта существовали
    # только в переменной и никуда не попадали.
    known = {name for _, name in names}
    orphans = sorted(set(totals) - known)
    if orphans:
        log.warning("Посчитан итог по %s, но в блоке «%s» для них нет "
                   "строки — допишите строку с именем проекта в лист "
                   "и запустите ещё раз, иначе эти суммы потеряются.",
                   ", ".join(orphans), month)

    payload, shown = [], []
    for row_no, name in names:
        vals = totals.get(name)
        if vals is None:
            log.warning("«%s»: данных за месяц нет, строка не тронута", name)
            shown.append({"Проект": name, **{m: "—" for m in METRIC_NAMES}})
            continue

        def cast(x):
            return int(x) if float(x).is_integer() else round(float(x), 2)

        if WRITE_CRM_COLUMN:
            payload.append({"range": f"B{row_no}:F{row_no}",
                            "values": [[cast(vals[i]) for i in range(5)]]})
        else:
            payload.append({"range": f"B{row_no}:C{row_no}",
                            "values": [[cast(vals[0]), cast(vals[1])]]})
            payload.append({"range": f"E{row_no}:F{row_no}",
                            "values": [[cast(vals[3]), cast(vals[4])]]})
        shown.append({"Проект": name,
                      **{m: cast(v) for m, v in zip(METRIC_NAMES, vals)}})

    if payload:
        ws.batch_update(payload, value_input_option="USER_ENTERED")
    log.info("Лист «%s», блок «%s»: обновлено строк %d",
             ws.title, month, len(payload))
    return pd.DataFrame(shown)


totals, month_report, loader_m, M_FROM, M_TO, PER_DAY = collect_month(REPORT_DATE)

print(f"\nМесяц: {M_FROM.strftime('%d.%m.%Y')} — {M_TO.strftime('%d.%m.%Y')}")
print(f"Листов с данными: {month_report['с данными']} из {month_report['листов найдено']}")

# Вклад каждого дня и накопительный итог — чтобы месячную сумму
# можно было проверить глазами, а не принимать на веру
day_rows = []
running = {}
for day in sorted(PER_DAY):
    vals = PER_DAY[day]["значения"]
    row = {"Дата": day.strftime("%d.%m"), "Строк": PER_DAY[day]["строк"]}
    for i, m in enumerate(METRIC_NAMES[:2] + METRIC_NAMES[3:5]):
        idx = i if i < 2 else i + 1
        s_day = sum(v[idx] for v in vals.values()
                    if idx < len(v) and pd.notna(v[idx]))
        running[m] = running.get(m, 0) + s_day
        row[m] = int(s_day)
        row[f"{m} всего"] = int(running[m])
    day_rows.append(row)
preview(pd.DataFrame(day_rows), "ВКЛАД КАЖДОГО ДНЯ И НАКОПИТЕЛЬНЫЙ ИТОГ")

if month_report["пустых"]:
    print(f"\nЛист есть, но блок I:O пустой: {', '.join(month_report['пустых'])}")
if month_report["нет листа"]:
    print(f"\nВНИМАНИЕ. За эти дни листа нет вообще, "
          f"в месячный итог они не вошли:")
    print(f"   {', '.join(month_report['нет листа'])}")
if month_report["сломанные формулы"]:
    print(f"\nВНИМАНИЕ. Ячейки с ошибкой формулы — эти значения пропущены:")
    for day, name, metric, cell in month_report["сломанные формулы"][:20]:
        print(f"   {day.strftime('%d.%m')} {name} / {metric}: {cell}")
if month_report["не распознаны"]:
    print(f"\nВНИМАНИЕ. Имя вкладки похоже на дату, но прочитать его "
          f"как дату не удалось — эти дни в сумму НЕ попали:")
    for t in month_report["не распознаны"]:
        print(f"   «{t}» — ожидается формат ДД.ММ.ГГ, например 01.09.26")

if month_report["расхождения"]:
    print("\nСВОДНЫЙ БЛОК I:O НЕ СХОДИТСЯ С ПОЧАСОВОЙ СЕТКОЙ A:G:")
    print(f"   {'Проект':<20}{'Показатель':<14}{'блок I:O':>10}{'сетка A:G':>11}{'разница':>9}")
    for name, metric, a, b in month_report["расхождения"]:
        print(f"   {name:<20}{metric:<14}{a:>10}{b:>11}{a - b:>9}")
    raise AssertionError(
        "Расхождение между сводным блоком и почасовой сеткой. "
        "Скорее всего на каком-то дневном листе сбита формула СУММЕСЛИ "
        "или в сетку попала лишняя строка. Итог не записан."
    )

print("\nСводный блок сходится с почасовой сеткой по всем проектам.")

month_df = write_month_total(totals, loader_m, REPORT_DATE)
preview(month_df, f"ИТОГ ЗА {RU_MONTHS[REPORT_DATE.month].upper()}")
print("\nНе тронуты: колонка «Итог» (там формула строки)"
      + ("" if WRITE_CRM_COLUMN else " и «CRM/Входящие» (заполняется вручную)")
      + ".")